# Role-Based Cybersecurity Risk Prediction
## Professional Machine Learning Training Pipeline

**Project objective:** Train a supervised machine-learning model that predicts an employee's cybersecurity risk level—**Low, Medium, or High**—using organizational context, employee role, access privileges, security controls, awareness behaviour, vulnerabilities and incident history.

This notebook demonstrates the complete ML-engineering workflow: data loading, data-quality checks, exploratory analysis, feature selection, leakage prevention, group-aware splitting, preprocessing, normalization, baseline testing, model comparison, hyperparameter tuning, evaluation, explainability and final `.pkl` export.

> Important research statement: the supplied dataset is synthetic. The resulting model is suitable for academic experimentation and prototype demonstration, not immediate real-world security decisions without external validation.

## 1. Machine-Learning Problem Definition

- **Learning type:** Supervised multiclass classification
- **Target variable:** `risk_level`
- **Classes:** `Low`, `Medium`, `High`
- **Primary success metric:** Macro F1-score
- **Supporting metrics:** Accuracy, balanced accuracy, macro precision and macro recall
- **Deployment output:** A serialized Scikit-learn pipeline stored as a `.pkl` file

Macro F1 is emphasized because it gives equal importance to all three risk classes instead of allowing the largest class to dominate the result.

## 2. Predictor Variable Groups

The model uses the following legitimate information groups:

1. **Organizational context** — industry, organization size, security budget, remote-work percentage and digital dependency.
2. **Role and access exposure** — employee role, department, seniority, work mode, privileged access and sensitive-system access.
3. **Security controls** — MFA, firewall, EDR, encryption, VPN, logging, backups, least privilege and access reviews.
4. **Awareness and behaviour** — training score, phishing behaviour, reporting rate, password-manager usage and public Wi-Fi usage.
5. **Technical weaknesses** — patch compliance, patch delay, vulnerabilities, unsupported software and device compliance.
6. **Incident history** — security, phishing and malware incidents, account compromise and data-breach history.

Identifiers and outputs calculated after risk assessment are excluded to prevent **target leakage**.

In [ ]:
# Install the few packages that may not already be available in Google Colab.
# Scikit-learn, pandas and NumPy are normally preinstalled.
!pip install -q openpyxl joblib seaborn

In [ ]:
# Core data, visualization and utility libraries
import hashlib
import json
import random
import warnings
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Scikit-learn components
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    RandomizedSearchCV,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from google.colab import files

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Reproducibility: the same seed helps reproduce the same split and model.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print('Environment initialized successfully.')

## 3. Upload and Load the Excel Dataset

Run the next cell and select `Role_Based_Cybersecurity_Risk_Dataset_5500.xlsx`. Only the `Employee_Risk_Data` sheet is used for model training; the other sheets support the recommendation engine and documentation.

In [ ]:
# Upload the Excel workbook from your computer to the Colab session.
uploaded = files.upload()

# Automatically use the first uploaded .xlsx file.
excel_files = [name for name in uploaded.keys() if name.lower().endswith('.xlsx')]
if not excel_files:
    raise FileNotFoundError('Please upload the generated .xlsx dataset.')

DATASET_PATH = excel_files[0]
SHEET_NAME = 'Employee_Risk_Data'

df = pd.read_excel(DATASET_PATH, sheet_name=SHEET_NAME, engine='openpyxl')
print(f'Loaded file: {DATASET_PATH}')
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())

## 4. Data Quality and Schema Audit

Before training, confirm the schema, missing values, duplicate records, role coverage and target-class distribution. This provides evidence that the dataset was reviewed instead of being passed directly into an algorithm.

In [ ]:
TARGET_COLUMN = 'risk_level'
REQUIRED_COLUMNS = {
    'record_id', 'organization_id', 'employee_role', TARGET_COLUMN
}

missing_required = REQUIRED_COLUMNS.difference(df.columns)
if missing_required:
    raise ValueError(f'Missing required columns: {sorted(missing_required)}')

audit_summary = pd.DataFrame({
    'measure': [
        'Rows', 'Columns', 'Duplicate rows',
        'Unique organizations', 'Unique employee roles',
        'Total missing cells'
    ],
    'value': [
        len(df), df.shape[1], df.duplicated().sum(),
        df['organization_id'].nunique(), df['employee_role'].nunique(),
        int(df.isna().sum().sum())
    ]
})
display(audit_summary)

print('Risk-level distribution:')
display(df[TARGET_COLUMN].value_counts().rename_axis('risk_level').to_frame('records'))

print('Employee-role distribution:')
display(df['employee_role'].value_counts().rename_axis('employee_role').to_frame('records'))

print('Columns with the most missing values:')
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame('missing_values'))

In [ ]:
# Visualize the target classes and role distribution for the thesis/report.
risk_order = ['Low', 'Medium', 'High']
risk_palette = {'Low': '#22C55E', 'Medium': '#F59E0B', 'High': '#EF4444'}

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.countplot(
    data=df, x=TARGET_COLUMN, order=risk_order, hue=TARGET_COLUMN,
    palette=risk_palette, legend=False, ax=axes[0]
)
axes[0].set_title('Cybersecurity Risk-Level Distribution')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Number of Employees')

role_order = df['employee_role'].value_counts().index
sns.countplot(data=df, y='employee_role', order=role_order, color='#0F766E', ax=axes[1])
axes[1].set_title('Employee Records by Role')
axes[1].set_xlabel('Number of Employees')
axes[1].set_ylabel('Employee Role')

plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Feature Selection and Target-Leakage Prevention

The following columns must not enter the model:

- `record_id` and `organization_id` are identifiers, not security characteristics.
- `assessment_date` and `data_source` are metadata.
- `overall_risk_score`, `role_risk_score` and `risk_level` are assessment results.
- `primary_risk_category`, `recommendation_priority`, `recommended_action_codes` and `training_focus_area` are post-prediction outputs.

Using these outputs as predictors would allow the model to see information derived from the correct answer and would produce misleadingly high performance.

In [ ]:
# Columns excluded from model input.
EXCLUDED_COLUMNS = [
    'record_id',
    'organization_id',
    'assessment_date',
    'data_source',
    'overall_risk_score',
    'role_risk_score',
    'risk_level',
    'primary_risk_category',
    'recommendation_priority',
    'recommended_action_codes',
    'training_focus_area',
]

# X contains predictors; y contains the label to be learned.
X = df.drop(columns=EXCLUDED_COLUMNS).copy()
y = df[TARGET_COLUMN].astype(str).copy()

# Organization ID is retained only as a grouping variable for safe splitting.
groups = df['organization_id'].astype(str).copy()

print(f'Predictor variables: {X.shape[1]}')
print(f'Target variable: {TARGET_COLUMN}')
print(f'Target classes: {sorted(y.unique())}')
display(pd.DataFrame({'model_input_column': X.columns}))

## 6. Group-Aware Train/Test Split

Employees from the same organization can share controls such as firewall, logging, backup and security policy. A normal random split could place employees from one organization into both training and testing sets, making the test result too optimistic.

`GroupShuffleSplit` keeps each organization entirely in one side of the split. The model is trained using 80% of organizations and independently tested using the remaining 20%.

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=RANDOM_STATE
)

train_index, test_index = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_index].copy()
X_test = X.iloc[test_index].copy()
y_train = y.iloc[train_index].copy()
y_test = y.iloc[test_index].copy()
train_groups = groups.iloc[train_index].copy()
test_groups = groups.iloc[test_index].copy()

# Confirm that no organization appears in both partitions.
organization_overlap = set(train_groups).intersection(set(test_groups))
assert len(organization_overlap) == 0, 'Organization leakage detected.'

split_summary = pd.DataFrame({
    'partition': ['Training', 'Testing'],
    'records': [len(X_train), len(X_test)],
    'organizations': [train_groups.nunique(), test_groups.nunique()]
})
display(split_summary)

print('Training target distribution:')
display(y_train.value_counts(normalize=True).mul(100).round(2).to_frame('percentage'))
print('Testing target distribution:')
display(y_test.value_counts(normalize=True).mul(100).round(2).to_frame('percentage'))

## 7. Preprocessing and Normalization Pipeline

The preprocessing pipeline is fitted only on training data during each validation fold. This avoids information from the test set influencing the training process.

- Missing numerical values: median imputation
- Numerical normalization: `StandardScaler`
- Missing categorical values: most-frequent imputation
- Categorical encoding: one-hot encoding
- Unknown website categories: safely ignored during inference

Normalization is particularly important for the Logistic Regression comparison model. Random Forest does not mathematically require scaling, but keeping preprocessing inside one pipeline provides a consistent deployment interface.

In [ ]:
# Automatically identify numerical and categorical predictor variables.
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

print(f'Numerical features: {len(numerical_features)}')
print(f'Categorical features: {len(categorical_features)}')

numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numerical', numerical_pipeline, numerical_features),
        ('categorical', categorical_pipeline, categorical_features),
    ],
    remainder='drop'
)

print('Preprocessing pipeline created successfully.')

## 8. Establish a Baseline

A baseline classifier predicts using class frequency rather than cybersecurity features. The trained model must clearly outperform this reference; otherwise, the model has not demonstrated meaningful learning.

In [ ]:
baseline_model = DummyClassifier(strategy='most_frequent')
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)

baseline_results = {
    'accuracy': accuracy_score(y_test, baseline_predictions),
    'balanced_accuracy': balanced_accuracy_score(y_test, baseline_predictions),
    'macro_f1': f1_score(y_test, baseline_predictions, average='macro', zero_division=0),
}

display(pd.DataFrame([baseline_results], index=['Dummy baseline']).round(4))

## 9. Compare Candidate Algorithms with Group Cross-Validation

Two suitable algorithms are compared:

- **Logistic Regression:** an interpretable linear classification benchmark.
- **Random Forest:** captures nonlinear relationships and interactions among roles, access levels, controls, behaviour and incidents.

Five-fold `GroupKFold` cross-validation keeps organizations separated in every fold. Model selection is based mainly on mean macro F1-score.

In [ ]:
candidate_models = {
    'Logistic Regression': LogisticRegression(
        max_iter=3000,
        class_weight='balanced',
        solver='saga',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=400,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
}

scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'macro_f1': 'f1_macro',
}

group_cv = GroupKFold(n_splits=5)
comparison_rows = []

for model_name, classifier in candidate_models.items():
    candidate_pipeline = Pipeline(steps=[
        ('preprocessor', clone(preprocessor)),
        ('classifier', classifier),
    ])

    cv_result = cross_validate(
        candidate_pipeline,
        X_train,
        y_train,
        groups=train_groups,
        cv=group_cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    comparison_rows.append({
        'model': model_name,
        'mean_accuracy': cv_result['test_accuracy'].mean(),
        'mean_balanced_accuracy': cv_result['test_balanced_accuracy'].mean(),
        'mean_macro_f1': cv_result['test_macro_f1'].mean(),
        'macro_f1_std': cv_result['test_macro_f1'].std(),
        'mean_fit_time_seconds': cv_result['fit_time'].mean(),
    })

model_comparison = pd.DataFrame(comparison_rows).sort_values(
    'mean_macro_f1', ascending=False
)
display(model_comparison.round(4))

## 10. Hyperparameter Tuning

Random Forest is selected as the deployment model because it handles nonlinear security relationships, mixed feature effects and complex role-based interactions. `RandomizedSearchCV` evaluates several configurations efficiently.

The tuning process adjusts the number and depth of trees, split constraints, feature sampling and class weighting. Group-aware folds continue to protect the experiment from organization leakage.

In [ ]:
random_forest_pipeline = Pipeline(steps=[
    ('preprocessor', clone(preprocessor)),
    ('classifier', RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    )),
])

parameter_distributions = {
    'classifier__n_estimators': [300, 450, 600, 750],
    'classifier__max_depth': [None, 12, 18, 24, 30],
    'classifier__min_samples_split': [2, 4, 6, 10],
    'classifier__min_samples_leaf': [1, 2, 3, 5],
    'classifier__max_features': ['sqrt', 'log2', 0.5],
    'classifier__class_weight': ['balanced', 'balanced_subsample'],
}

tuning_cv = GroupKFold(n_splits=4)

random_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=parameter_distributions,
    n_iter=12,
    scoring='f1_macro',
    cv=tuning_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
)

random_search.fit(X_train, y_train, groups=train_groups)

best_model = random_search.best_estimator_
print(f'Best cross-validated macro F1: {random_search.best_score_:.4f}')
print('Best hyperparameters:')
print(json.dumps(random_search.best_params_, indent=2))

## 11. Final Evaluation on Unseen Organizations

The test set was not used for preprocessing, model selection or tuning. It therefore provides the main estimate of performance on employees from previously unseen organizations.

In [ ]:
test_predictions = best_model.predict(X_test)
test_probabilities = best_model.predict_proba(X_test)

evaluation_metrics = {
    'accuracy': float(accuracy_score(y_test, test_predictions)),
    'balanced_accuracy': float(balanced_accuracy_score(y_test, test_predictions)),
    'macro_precision': float(precision_score(
        y_test, test_predictions, average='macro', zero_division=0
    )),
    'macro_recall': float(recall_score(
        y_test, test_predictions, average='macro', zero_division=0
    )),
    'macro_f1': float(f1_score(
        y_test, test_predictions, average='macro', zero_division=0
    )),
}

metric_table = pd.DataFrame({
    'Metric': [name.replace('_', ' ').title() for name in evaluation_metrics],
    'Value': list(evaluation_metrics.values())
})
display(metric_table.style.format({'Value': '{:.4f}'}))

print('Detailed classification report:')
print(classification_report(
    y_test, test_predictions, labels=['Low', 'Medium', 'High'], zero_division=0
))

if evaluation_metrics['macro_f1'] <= baseline_results['macro_f1']:
    raise RuntimeError('The trained model did not outperform the baseline.')

In [ ]:
# Confusion matrix: shows correct and incorrect predictions for each class.
class_order = ['Low', 'Medium', 'High']
matrix = confusion_matrix(y_test, test_predictions, labels=class_order)

plt.figure(figsize=(8, 6))
sns.heatmap(
    matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
    xticklabels=class_order, yticklabels=class_order
)
plt.title('Confusion Matrix: Role-Based Cybersecurity Risk Prediction')
plt.xlabel('Predicted Risk Level')
plt.ylabel('Actual Risk Level')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 12. Role-Level Fairness and Reliability Check

Overall accuracy alone can hide weak performance for a particular employee category. This section reports accuracy and macro F1 separately for each role to support the project's role-based requirement.

In [ ]:
role_evaluation = X_test[['employee_role']].copy()
role_evaluation['actual_risk'] = y_test.values
role_evaluation['predicted_risk'] = test_predictions

role_rows = []
for role, role_data in role_evaluation.groupby('employee_role'):
    role_rows.append({
        'employee_role': role,
        'test_records': len(role_data),
        'accuracy': accuracy_score(
            role_data['actual_risk'], role_data['predicted_risk']
        ),
        'macro_f1': f1_score(
            role_data['actual_risk'],
            role_data['predicted_risk'],
            average='macro',
            zero_division=0,
        ),
    })

role_performance = pd.DataFrame(role_rows).sort_values(
    'macro_f1', ascending=False
)
display(role_performance.style.format({'accuracy': '{:.4f}', 'macro_f1': '{:.4f}'}))

plt.figure(figsize=(12, 7))
sns.barplot(data=role_performance, x='macro_f1', y='employee_role', color='#0F766E')
plt.xlim(0, 1)
plt.title('Macro F1-Score by Employee Role')
plt.xlabel('Macro F1-Score')
plt.ylabel('Employee Role')
plt.tight_layout()
plt.savefig('role_based_performance.png', dpi=300, bbox_inches='tight')
plt.show()

## 13. Feature Importance and Model Explainability

Random Forest feature importance indicates which transformed variables contributed most to predictions. Importance does not prove causation, but it helps explain the model and verify that it relies on meaningful security characteristics.

In [ ]:
trained_preprocessor = best_model.named_steps['preprocessor']
trained_classifier = best_model.named_steps['classifier']

transformed_feature_names = trained_preprocessor.get_feature_names_out()
clean_feature_names = [
    name.replace('numerical__', '').replace('categorical__', '')
    for name in transformed_feature_names
]

feature_importance = pd.DataFrame({
    'feature': clean_feature_names,
    'importance': trained_classifier.feature_importances_,
}).sort_values('importance', ascending=False)

display(feature_importance.head(25))

top_features = feature_importance.head(20).sort_values('importance')
plt.figure(figsize=(12, 8))
plt.barh(top_features['feature'], top_features['importance'], color='#2563EB')
plt.title('Top 20 Features Influencing Cybersecurity Risk')
plt.xlabel('Random Forest Feature Importance')
plt.ylabel('Transformed Feature')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## 14. Prediction Confidence and Error Analysis

Prediction probabilities can be used by the website to display confidence or flag uncertain cases for additional review. Error analysis identifies the types of employees the model finds difficult to classify.

In [ ]:
model_classes = best_model.named_steps['classifier'].classes_
prediction_details = X_test[['employee_role', 'department']].copy()
prediction_details['actual_risk'] = y_test.values
prediction_details['predicted_risk'] = test_predictions
prediction_details['prediction_confidence'] = test_probabilities.max(axis=1)
prediction_details['correct_prediction'] = (
    prediction_details['actual_risk'] == prediction_details['predicted_risk']
)

for class_index, class_name in enumerate(model_classes):
    prediction_details[f'probability_{class_name.lower()}'] = (
        test_probabilities[:, class_index]
    )

print('Lowest-confidence predictions:')
display(prediction_details.sort_values('prediction_confidence').head(20))

print('Sample misclassified records:')
display(prediction_details.loc[
    ~prediction_details['correct_prediction']
].sort_values('prediction_confidence').head(20))

## 15. Save the Complete Model Package as a `.pkl` File

The exported package includes the complete preprocessing pipeline and classifier. The website therefore sends raw feature values in the original column format; imputation, normalization and one-hot encoding happen automatically inside the saved pipeline.

In [ ]:
MODEL_FILENAME = 'role_based_cybersecurity_risk_model.pkl'

model_package = {
    'model_name': 'Role-Based Cybersecurity Risk Prediction Model',
    'model_version': '1.0.0',
    'pipeline': best_model,
    'target_column': TARGET_COLUMN,
    'class_labels': model_classes.tolist(),
    'feature_columns': X.columns.tolist(),
    'numerical_features': numerical_features,
    'categorical_features': categorical_features,
    'excluded_columns': EXCLUDED_COLUMNS,
    'best_hyperparameters': random_search.best_params_,
    'evaluation_metrics': evaluation_metrics,
    'training_records': int(len(X_train)),
    'testing_records': int(len(X_test)),
    'training_organizations': int(train_groups.nunique()),
    'testing_organizations': int(test_groups.nunique()),
    'random_state': RANDOM_STATE,
    'trained_at_utc': datetime.now(timezone.utc).isoformat(),
    'data_notice': 'Trained on a synthetic academic research dataset.',
}

joblib.dump(model_package, MODEL_FILENAME, compress=3)

# Produce a checksum so the exact model file can be verified after transfer.
with open(MODEL_FILENAME, 'rb') as model_file:
    model_sha256 = hashlib.sha256(model_file.read()).hexdigest()

print(f'Model saved: {MODEL_FILENAME}')
print(f'SHA-256: {model_sha256}')
print(f'Macro F1: {evaluation_metrics["macro_f1"]:.4f}')

## 16. Reload and Validate the Saved Model

A deployment artifact should be tested after serialization. This cell reloads the `.pkl` file and confirms that it produces the same prediction as the in-memory model.

In [ ]:
loaded_package = joblib.load(MODEL_FILENAME)
loaded_pipeline = loaded_package['pipeline']

sample_input = X_test.iloc[[0]].copy()
original_prediction = best_model.predict(sample_input)[0]
reloaded_prediction = loaded_pipeline.predict(sample_input)[0]

assert original_prediction == reloaded_prediction
assert sample_input.columns.tolist() == loaded_package['feature_columns']

sample_probabilities = loaded_pipeline.predict_proba(sample_input)[0]
sample_result = pd.DataFrame({
    'risk_level': loaded_package['class_labels'],
    'probability': sample_probabilities
}).sort_values('probability', ascending=False)

print('Saved model validation passed.')
print(f'Sample role: {sample_input["employee_role"].iloc[0]}')
print(f'Predicted risk: {reloaded_prediction}')
display(sample_result)

## 17. Export Evaluation Evidence

The Excel report and PNG figures can be used in the results and discussion chapters of the thesis.

In [ ]:
with pd.ExcelWriter('model_training_results.xlsx', engine='openpyxl') as writer:
    metric_table.to_excel(writer, sheet_name='Overall_Metrics', index=False)
    model_comparison.to_excel(writer, sheet_name='Model_Comparison', index=False)
    role_performance.to_excel(writer, sheet_name='Role_Performance', index=False)
    feature_importance.to_excel(writer, sheet_name='Feature_Importance', index=False)
    prediction_details.to_excel(writer, sheet_name='Test_Predictions', index=False)

print('Created model_training_results.xlsx')
print('Created confusion_matrix.png')
print('Created role_based_performance.png')
print('Created feature_importance.png')

## 18. Download the Trained Model

Run the following cell after all previous cells complete successfully. Keep the `.pkl` file with the backend project and do not edit it manually.

In [ ]:
# Download the final model package to your computer.
files.download(MODEL_FILENAME)

# Run these separately if you also want the evaluation evidence.
# files.download('model_training_results.xlsx')
# files.download('confusion_matrix.png')
# files.download('role_based_performance.png')
# files.download('feature_importance.png')

## 19. Website/Backend Inference Pattern

The backend must provide all raw model-input columns. It should not manually normalize or encode them because those operations are already included in the saved pipeline. The recommendation engine should run **after** risk prediction and combine the predicted risk level, employee role and detected control gaps with `Recommendation_Rules`.

In [ ]:
# Demonstration of the code pattern that can later be used in a Python API.
deployment_package = joblib.load('role_based_cybersecurity_risk_model.pkl')
deployment_model = deployment_package['pipeline']
required_features = deployment_package['feature_columns']

# Replace this sample with one employee JSON payload from the website.
employee_payload = X_test.iloc[0].to_dict()
employee_frame = pd.DataFrame([employee_payload]).reindex(columns=required_features)

predicted_risk = deployment_model.predict(employee_frame)[0]
predicted_probabilities = deployment_model.predict_proba(employee_frame)[0]
confidence = float(predicted_probabilities.max())

api_response = {
    'employee_role': employee_payload['employee_role'],
    'predicted_risk_level': predicted_risk,
    'confidence': round(confidence, 4),
}

print(api_response)

## 20. Thesis Explanation of the Training Process

A supervised multiclass classification approach was used to predict employee cybersecurity risk as Low, Medium or High. The predictor set represented organizational context, role-specific access, implemented security controls, employee security behaviour, technical vulnerabilities and incident history. Identifiers, calculated risk scores and recommendation outputs were removed to prevent target leakage.

The dataset was divided using an organization-aware split so that employees from the same organization could not appear in both training and testing sets. Numerical variables were median-imputed and standardized, while categorical variables were imputed using the most frequent value and transformed through one-hot encoding. All preprocessing steps were contained within a Scikit-learn pipeline.

A frequency-based dummy classifier established the baseline. Logistic Regression and Random Forest were then compared through group-based cross-validation using macro F1-score, balanced accuracy and accuracy. Random Forest was selected and optimized through randomized hyperparameter search. Final performance was measured using previously unseen organizations, a classification report, confusion matrix and employee-role-level analysis. The preprocessing pipeline, classifier, feature schema, class labels, parameters and evaluation metadata were serialized into one deployable `.pkl` package for integration with the web platform.